# Session 29: Pipelines

Data Science with Python, Level 2, Stage 4: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Explain data leakage and how pipelines prevent it.
2. Build a Pipeline with named steps.
3. Select column groups automatically.
4. Write custom transformers.
5. Change steps and settings with set_params.
6. Use one pipeline for training and new data.

## Where We Are

Since Session 22 you have used `make_pipeline` to keep scaling inside cross-validation. Over the last sessions the steps have grown: imputing, encoding, scaling, sometimes new features, then a model.

Today the pipeline becomes the centre of the work. You will see how leakage can make a useless model look good, build pipelines with named steps, write your own transformer for new features, and swap models with one line. The same pipeline object is what you will save and put into a web app in Stage 5.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)
X = churn.drop(columns=["customer_id", "churn"])

## 1. Leakage: The Silent Bug

In [ ]:
rng = np.random.default_rng(1)
noise = rng.normal(size=(200, 2000))
target = rng.integers(0, 2, size=200)

In [ ]:
best20 = SelectKBest(f_classif, k=20).fit(noise, target)
leaky = cross_val_score(LogisticRegression(), best20.transform(noise), target, cv=5)

safe_model = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
safe = cross_val_score(safe_model, noise, target, cv=5)
print("Select first, then validate:", leaky.mean().round(3))
print("Select inside the pipeline: ", safe.mean().round(3))

## 2. Pipelines with Names

In [ ]:
num_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
cat_steps = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                          OneHotEncoder(handle_unknown="ignore"))

preprocess = ColumnTransformer([
    ("num", num_steps, make_column_selector(dtype_include="number")),
    ("cat", cat_steps, make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])

In [ ]:
pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])
auc = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print("AUC:", auc.mean().round(3))
pipe.fit(X, y)
print(pipe.named_steps["model"].coef_.shape)

In [ ]:
pipe.set_params(model=HistGradientBoostingClassifier(
    learning_rate=0.05, max_depth=3, random_state=0))
auc = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print("Boosting AUC:", auc.mean().round(3))

print(pipe.get_params()["prep__num__simpleimputer__strategy"])

## 3. Custom Transformers

In [ ]:
log_step = FunctionTransformer(np.log1p, feature_names_out="one-to-one")
print(log_step.fit_transform(X[["total_charges"]]).head(3))

In [ ]:
class AddChurnFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["calls_per_year"] = X["support_calls"] / (X["tenure_months"] / 12 + 1)
        X["new_customer"] = X["tenure_months"] < 6
        return X

In [ ]:
pipe = Pipeline([
    ("features", AddChurnFeatures()),
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])
auc = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print("AUC:", auc.mean().round(3))
pipe.fit(X, y)
print(len(pipe["prep"].get_feature_names_out()), "columns")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
pipe.fit(X_train, y_train)

new = X_test.head(3)
print(new[["tenure_months", "contract", "monthly_charges"]])
print(pipe.predict_proba(new)[:, 1].round(2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Scaling outside | `StandardScaler().fit_transform(X)` before `cross_val_score` | Put the scaler inside the pipeline. |
| Wrong param name | `pipe.set_params(C=1)` | Use the step name: `model__C=1`. |
| Changing input | A transformer that edits `X` in place | Work on `X.copy()` in `transform`. |
| Fit in transform | Learning means inside `transform` | Learn in `fit`, store with a trailing `_`, apply in `transform`. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Spot the Leak

Explain in a comment which of these leaks: (a) `StandardScaler().fit(X)` then `train_test_split`; (b) `train_test_split` then a pipeline with a scaler; (c) dropping rows with missing values before splitting; (d) choosing features by their correlation with the target on the full data. Then print your answers.

In [ ]:
# Write your code below

### In-Class Exercise 2: Three Models, One Pipeline

Using `pipe` with the `features` and `prep` steps, loop over a logistic regression, a random forest (300 trees, `min_samples_leaf=10`) and a boosted model. Use `set_params(model=...)` and print the cross-validated AUC for each.

In [ ]:
# Write your code below

### In-Class Exercise 3: Log Charges

Change the numeric part of the preprocessor to impute, apply `np.log1p`, then scale. Does the cross-validated AUC of the logistic pipeline change?

In [ ]:
# Write your code below

### In-Class Exercise 4: A Learning Transformer

Write a transformer `ClipOutliers` that learns the 1st and 99th percentile of each numeric column in `fit` (store them as `self.low_` and `self.high_`) and clips values to that range in `transform`. Test it on `X[["monthly_charges", "total_charges"]]`.

In [ ]:
# Write your code below

## Summary

- Leakage lets test data influence training and inflates scores.
- Everything that learns from data belongs inside the pipeline.
- `Pipeline` names steps; `named_steps` and `set_params` reach them.
- `make_column_selector` picks columns by type automatically.
- Custom transformers need `fit` and `transform`.
- A fitted pipeline turns raw rows into predictions in one call.

## What's Next

Every model has settings: depth, learning rate, regularisation, number of trees. Next session you will search for the best combination with GridSearchCV and RandomizedSearchCV, using the step__setting names from today.